# sphinx_design

sphinx-design 0.7.0 adds responsive page components to Sphinx: grids and cards, tab sets, dropdowns, badges and
buttons, written in reStructuredText or in MyST Markdown. These cells build small projects in a temporary folder and
check the HTML for each component, including a MyST page and a strict build that rejects a misplaced tab item.

In [ ]:
# Example 1: a two-column grid of cards, and the extension's CSS copied into the build.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_design']\n")
(src / "index.rst").write_text("Cards\n=====\n\n.. grid:: 2\n\n   .. grid-item-card:: Left\n\n      left body\n\n   .. grid-item-card:: Right\n\n      right body\n")
rc = build_main(["-q", "-W", "-n", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
css = list((src / "_out" / "_static").glob("sphinx-design*.css"))
print("rc", rc, "cards", html.count('class="sd-card '), css)
assert rc == 0
assert html.count('class="sd-card ') == 2 and "left body" in html and "right body" in html
assert css, "the extension's stylesheet was not copied"
from sphinx_design.shared import margin_option, text_align
assert margin_option("1") == ["sd-m-1"] and text_align("center") == ["sd-text-center"]  # the helpers every component's options go through

In [ ]:
# Example 2: a tab set: one tab item per label, the first one selected.
import pathlib, re, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_design']\n")
(src / "index.rst").write_text("Tabs\n====\n\n.. tab-set::\n\n   .. tab-item:: pip\n\n      pip text\n\n   .. tab-item:: uv\n\n      uv text\n")
assert build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")]) == 0
html = (src / "_out" / "index.html").read_text()
labels = re.findall(r'<label class="sd-tab-label"[^>]*>\s*([^<]+?)\s*</label>', html)
print(labels)
assert labels == ["pip", "uv"], labels
assert html.count('type="radio"') == 2 and "checked" in html

In [ ]:
# Example 4: the same components from MyST Markdown (python/myst_parser), with colon fences.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['myst_parser', 'sphinx_design']\nmyst_enable_extensions = ['colon_fence']\n")
(src / "index.md").write_text(
    "# MyST\n\n::::{tab-set}\n:::{tab-item} One\nfirst\n:::\n:::{tab-item} Two\nsecond\n:::\n::::\n\n"
    "::::{grid} 2\n:::{grid-item-card} A\nalpha\n:::\n:::{grid-item-card} B\nbeta\n:::\n::::\n")
rc = build_main(["-q", "-W", "-n", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
print("rc", rc, "tab sets", html.count('class="sd-tab-set'), "cards", html.count('class="sd-card '))
assert rc == 0
assert html.count('class="sd-tab-set') == 1 and "second" in html
assert html.count('class="sd-card ') == 2 and "beta" in html

In [ ]:
# Example 5: a strict build rejects a tab item outside a tab set, naming the rule.
import io, pathlib, tempfile
from contextlib import redirect_stderr
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_design']\n")
(src / "index.rst").write_text("Bad\n===\n\n.. tab-item:: orphan\n\n   no parent\n")
err = io.StringIO()
with redirect_stderr(err):
    rc = build_main(["-q", "-W", "--keep-going", "-b", "html", str(src), str(src / "_out")])
print("rc", rc, err.getvalue()[-300:])
assert rc == 1
assert "The parent of a 'tab-item' should be a 'tab-set'" in err.getvalue()